# Testing SQL Agent


## Demo simple de un agente SQL para la base de datos del videoclub

Este notebook recibe una consulta en lenguaje natural, genera una sentencia SQL compatible con PostgreSQL y, si la consulta es segura, la ejecuta sobre la base de datos.


In [ ]:
!python -m pip install -r requirements.txt


In [1]:
import json
import os
import re

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set.")

client = OpenAI(api_key=api_key)
print("OPENAI_API_KEY successfully loaded.")


OPENAI_API_KEY successfully loaded.


In [3]:
# Parametros de conexion. Si existen variables de entorno, tienen prioridad.
DB_HOST = os.getenv("DB_HOST", "46.183.117.210")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "esesa")
DB_USER = os.getenv("DB_USER", "esesa")
DB_PASSWORD = os.getenv("DB_PASSWORD", "esesa")

print(f"Database host loaded: {DB_HOST}")


Database host loaded: 46.183.117.210


In [5]:
def connect_to_db():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
        )
        print("Connection successful")
        return conn
    except Exception as e:
        print(f"Error connecting to database: {e}")
        return None


def is_safe_sql(sql_text):
    normalized_sql = sql_text.strip().lower()
    forbidden_keywords = [
        "insert ",
        "update ",
        "delete ",
        "drop ",
        "alter ",
        "truncate ",
        "create ",
        "grant ",
        "revoke ",
    ]
    starts_as_read_only = normalized_sql.startswith("select") or normalized_sql.startswith("with")
    has_forbidden_keywords = any(keyword in normalized_sql for keyword in forbidden_keywords)
    return starts_as_read_only and not has_forbidden_keywords


def execute_select_query(query):
    if not is_safe_sql(query):
        raise ValueError("Solo se permiten consultas SELECT o WITH de solo lectura.")

    conn = connect_to_db()
    if conn is None:
        return None

    try:
        df = pd.read_sql_query(query, conn)
        return df
    finally:
        conn.close()


In [6]:
schema_context = """
Base de datos del videoclub (PostgreSQL):

Tablas principales:
- customer(customer_id, store_id, first_name, last_name, email, address_id, active, create_date)
- film(film_id, title, description, release_year, language_id, rental_duration, rental_rate, length, rating)
- inventory(inventory_id, film_id, store_id)
- rental(rental_id, rental_date, inventory_id, customer_id, return_date, staff_id)
- payment(payment_id, customer_id, staff_id, rental_id, amount, payment_date)
- category(category_id, name)
- film_category(film_id, category_id)
- actor(actor_id, first_name, last_name)
- film_actor(actor_id, film_id)
- store(store_id, manager_staff_id, address_id)
- address(address_id, address, district, city_id, postal_code, phone)
- city(city_id, city, country_id)
- country(country_id, country)
- staff(staff_id, first_name, last_name, email, store_id, active)

Relaciones utiles:
- rental.inventory_id -> inventory.inventory_id
- inventory.film_id -> film.film_id
- rental.customer_id -> customer.customer_id
- payment.rental_id -> rental.rental_id
- payment.customer_id -> customer.customer_id
- film_category.film_id -> film.film_id
- film_category.category_id -> category.category_id
- film_actor.film_id -> film.film_id
- film_actor.actor_id -> actor.actor_id
- customer.address_id -> address.address_id
- address.city_id -> city.city_id
- city.country_id -> country.country_id

Reglas:
- Genera solo SQL de lectura.
- Usa LIMIT 10 salvo que el usuario pida otro limite.
- Si la peticion implica agregaciones, añade alias claros.
- Usa JOIN explicitos.
- Si filtras clientes, prioriza customer.active = true cuando tenga sentido.
"""

print(schema_context)



Base de datos del videoclub (PostgreSQL):

Tablas principales:
- customer(customer_id, store_id, first_name, last_name, email, address_id, active, create_date)
- film(film_id, title, description, release_year, language_id, rental_duration, rental_rate, length, rating)
- inventory(inventory_id, film_id, store_id)
- rental(rental_id, rental_date, inventory_id, customer_id, return_date, staff_id)
- payment(payment_id, customer_id, staff_id, rental_id, amount, payment_date)
- category(category_id, name)
- film_category(film_id, category_id)
- actor(actor_id, first_name, last_name)
- film_actor(actor_id, film_id)
- store(store_id, manager_staff_id, address_id)
- address(address_id, address, district, city_id, postal_code, phone)
- city(city_id, city, country_id)
- country(country_id, country)
- staff(staff_id, first_name, last_name, email, store_id, active)

Relaciones utiles:
- rental.inventory_id -> inventory.inventory_id
- inventory.film_id -> film.film_id
- rental.customer_id -> custom

In [7]:
system_prompt = f"""
Eres un asistente que traduce preguntas de negocio a SQL para PostgreSQL.

{schema_context}

Debes devolver un JSON valido con esta estructura exacta:
{{
  \"sql\": \"consulta SQL\",
  \"explanation\": \"explicacion breve en espanol\"
}}

No incluyas markdown ni bloques de codigo.
"""


def clean_json_response(raw_text):
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```json\\s*", "", cleaned)
    cleaned = re.sub(r"^```\\s*", "", cleaned)
    cleaned = re.sub(r"\\s*```$", "", cleaned)
    return cleaned.strip()


def generate_sql(user_question, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    )

    content = response.choices[0].message.content
    payload = json.loads(clean_json_response(content))
    return payload


def ask_sql_agent(user_question, run_query=True):
    result = generate_sql(user_question)
    sql_query = result["sql"].strip()
    explanation = result["explanation"].strip()

    display(Markdown(f"**Pregunta:** {user_question}"))
    display(Markdown(f"**Explicacion:** {explanation}"))
    display(Markdown("**SQL generado:**"))
    display(Markdown(f"```sql\n{sql_query}\n```"))

    if not run_query:
        return sql_query, None

    query_result = execute_select_query(sql_query)
    return sql_query, query_result


In [8]:
user_question = "Cuales son las 10 peliculas mas alquiladas y cuantas veces se ha alquilado cada una?"
generated_sql, result_df = ask_sql_agent(user_question)

if result_df is not None:
    result_df.head(10)


**Pregunta:** Cuales son las 10 peliculas mas alquiladas y cuantas veces se ha alquilado cada una?

**Explicacion:** Esta consulta selecciona los títulos de las películas y cuenta cuántas veces se han alquilado, agrupando por título y ordenando de mayor a menor cantidad de alquileres, limitando el resultado a las 10 más alquiladas.

**SQL generado:**

```sql
SELECT f.title, COUNT(r.rental_id) AS total_alquileres FROM film f JOIN inventory i ON f.film_id = i.film_id JOIN rental r ON i.inventory_id = r.inventory_id GROUP BY f.title ORDER BY total_alquileres DESC LIMIT 10;
```

Connection successful


/var/folders/l3/mbsbyvn10fx1k145n7_7lvn80000gn/T/ipykernel_19515/1201836614.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [9]:
# Cambia la pregunta y vuelve a ejecutar esta celda.
another_question = "Que categorias generan mas ingresos en pagos?"
generated_sql, result_df = ask_sql_agent(another_question)

if result_df is not None:
    result_df.head(10)


**Pregunta:** Que categorias generan mas ingresos en pagos?

**Explicacion:** Esta consulta suma los ingresos totales de cada categoría de películas a partir de los pagos realizados, agrupando por el nombre de la categoría y ordenando los resultados de mayor a menor ingreso.

**SQL generado:**

```sql
SELECT c.name AS category_name, SUM(p.amount) AS total_income FROM payment p JOIN rental r ON p.rental_id = r.rental_id JOIN inventory i ON r.inventory_id = i.inventory_id JOIN film_category fc ON i.film_id = fc.film_id JOIN category c ON fc.category_id = c.category_id GROUP BY c.name ORDER BY total_income DESC LIMIT 10;
```

Connection successful


/var/folders/l3/mbsbyvn10fx1k145n7_7lvn80000gn/T/ipykernel_19515/1201836614.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
